In [8]:
%%writefile mm_input.txt
A,0,0,1
A,0,1,2
A,1,0,3
A,1,1,4
B,0,0,5
B,0,1,6
B,1,0,7
B,1,1,8

Overwriting mm_input.txt


In [9]:
%%writefile mapper_mm.py
import sys

for line in sys.stdin:
    matrix, i, j, value = line.strip().split(',')
    i, j = int(i), int(j)

    if matrix == 'A':
        for k in range(2):
            print(f"{i},{k}\tA,{j},{value}")
    else:
        for k in range(2):
            print(f"{k},{j}\tB,{i},{value}")

Overwriting mapper_mm.py


In [10]:
%%writefile reducer_mm.py
import sys


def emit_result(key, values):
    a_terms = {}
    b_terms = {}

    for item in values:
        parts = item.split(',')
        if len(parts) != 3:
            continue

        tag, idx, val = parts
        try:
            idx = int(idx)
            val = float(val)
        except ValueError:
            continue

        if tag == 'A':
            a_terms[idx] = val
        elif tag == 'B':
            b_terms[idx] = val

    all_k = set(a_terms) | set(b_terms)
    result = sum(a_terms.get(k, 0.0) * b_terms.get(k, 0.0) for k in all_k)

    if result.is_integer():
        result_text = str(int(result))
    else:
        result_text = f"{result:.6f}".rstrip('0').rstrip('.')

    print(f"{key}\t{result_text}")


current_key = None
bucket = []

for raw in sys.stdin:
    line = raw.strip()
    if not line:
        continue

    if '\t' not in line:
        continue

    key, val = line.split('\t', 1)

    if current_key is None:
        current_key = key

    if key != current_key:
        emit_result(current_key, bucket)
        current_key = key
        bucket = []

    bucket.append(val)

if current_key is not None:
    emit_result(current_key, bucket)

Overwriting reducer_mm.py


In [11]:
%%bash
set -e

if ! command -v java >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq openjdk-11-jdk-headless
fi

JAVA_BIN="$(readlink -f "$(command -v java)")"
JAVA_HOME_DIR="$(dirname "$(dirname "$JAVA_BIN")")"

if command -v hadoop >/dev/null 2>&1; then
  HADOOP_BIN="$(command -v hadoop)"
  HADOOP_HOME="$(cd "$(dirname "$HADOOP_BIN")/.." && pwd)"
else
  HADOOP_VERSION=3.3.6
  if [ ! -d "$HOME/hadoop-$HADOOP_VERSION" ]; then
    wget -q "https://archive.apache.org/dist/hadoop/common/hadoop-$HADOOP_VERSION/hadoop-$HADOOP_VERSION.tar.gz"
    tar -xzf "hadoop-$HADOOP_VERSION.tar.gz" -C "$HOME"
  fi
  HADOOP_HOME="$HOME/hadoop-$HADOOP_VERSION"
fi

echo "export JAVA_HOME=$JAVA_HOME_DIR" > hadoop_env.sh
echo "export HADOOP_HOME=$HADOOP_HOME" >> hadoop_env.sh
echo 'export PATH=$HADOOP_HOME/bin:$PATH' >> hadoop_env.sh

source hadoop_env.sh
hadoop version | head -n 1

Hadoop 3.3.6


In [12]:
%%bash
set -e

if [ -f hadoop_env.sh ]; then
  source hadoop_env.sh
fi

if ! command -v hadoop >/dev/null 2>&1; then
  echo "Hadoop not found. Run Cell 4 (Hadoop setup) first." >&2
  exit 1
fi

PYTHON_BIN=$(command -v python3 || true)
if [ -z "$PYTHON_BIN" ]; then
  PYTHON_BIN=$(command -v python || true)
fi
if [ -z "$PYTHON_BIN" ]; then
  echo "Python interpreter not found for Hadoop Streaming mapper/reducer." >&2
  exit 1
fi

STREAMING_JAR=$(ls "$HADOOP_HOME"/share/hadoop/tools/lib/hadoop-streaming*.jar 2>/dev/null | head -n 1)
if [ -z "$STREAMING_JAR" ]; then
  echo "Could not find hadoop-streaming jar under $HADOOP_HOME/share/hadoop/tools/lib" >&2
  exit 1
fi

rm -rf output
LOG_FILE=mm_hadoop.log
set +e
hadoop jar "$STREAMING_JAR" \
  -D mapreduce.framework.name=local \
  -files mapper_mm.py,reducer_mm.py \
  -cmdenv PATH="$PATH" \
  -input mm_input.txt \
  -output output \
  -mapper "$PYTHON_BIN mapper_mm.py" \
  -reducer "$PYTHON_BIN reducer_mm.py" > "$LOG_FILE" 2>&1
STATUS=$?
set -e

if [ $STATUS -ne 0 ]; then
  echo "Hadoop job failed. Filtered mapper/reducer errors:" >&2
  grep -E "Traceback|Error|Exception|No such file|command not found|PipeMapRed|subprocess failed" "$LOG_FILE" | tail -n 120 >&2 || true
  echo "--- Last 120 lines ---" >&2
  tail -n 120 "$LOG_FILE" >&2
  exit $STATUS
fi

echo "Matrix multiplication output:"
cat output/part-* | sort

Matrix multiplication output:
0,0	19
0,1	22
1,0	43
1,1	50
